# Session 19: Reinforcement Learning with Verifiable Rewards (RLVR)

In this notebook, we'll explore **RLVR** — a training paradigm that replaces human preference labels with automatic, rule-based reward signals. We'll apply it to our Personal Wellness Assistant to train a model that gives safer, more accurate wellness advice.

**Learning Objectives:**
- Understand the difference between RLHF and RLVR
- Define verifiable reward functions for a wellness use case
- Score model outputs automatically using reward functions
- Understand the GRPO training loop at a conceptual level
- Run a minimal RLVR training loop on a small language model
- Evaluate before and after training

## Table of Contents

- **Breakout Room #1: Understanding RLVR**
  - Task 1: Dependencies
  - Task 2: What Makes a Reward "Verifiable"?
  - Task 3: Defining a Verifiable Reward Function for Wellness
  - Task 4: Scoring Model Outputs
  - Task 5: The GRPO Training Loop
  - Question #1 & Question #2
  - 🏗️ Activity #1: Build and Test a Wellness Reward Function

- **Breakout Room #2: Training and Evaluation**
  - Task 6: Setting Up a Small Language Model
  - Task 7: Generating Rollouts
  - Task 8: Computing Rewards and Advantages
  - Task 9: Running a GRPO Update Step
  - Task 10: Evaluating Before and After Training
  - Question #3 & Question #4
  - 🏗️ Activity #2: Compare Base vs RLVR-Trained Wellness Responses

---
# 🤝 Breakout Room #1
## Understanding RLVR

## Task 1: Dependencies

Before we begin, make sure you have:

<!-- TODO: Confirm compute setup with Chris. -->
<!-- Options: Google Colab (free T4 GPU), DGX Spark, or local GPU. -->
<!-- Prerequisites will change depending on the compute environment. -->

1. A GPU environment (Google Colab T4 recommended if you don't have a local GPU)
2. The following packages installed

In [ ]:
# Install dependencies
# !pip install torch transformers datasets trl accelerate peft bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import json
import re

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Task 2: What Makes a Reward "Verifiable"?

### RLHF vs RLVR

In **RLHF (Reinforcement Learning from Human Feedback)**:
- A human compares two model responses and picks the better one
- A reward *model* is trained on those preferences
- The LLM is then trained to maximize that learned reward
- Problem: expensive, slow, subjective, and the reward model can be fooled

In **RLVR (Reinforcement Learning with Verifiable Rewards)**:
- Instead of a learned reward model, you use a **rule** or **function**
- The function checks something objectively true: is the answer correct? does it follow the format? does it avoid allergens?
- Reward is 1 if the rule is satisfied, 0 if not
- No human labeling needed at inference time

### The Key Insight

Not all tasks have verifiable rewards. But many do:

| Task | Verifiable Reward? | Why? |
|------|--------------------|------|
| Math problems | ✅ Yes | Answer is right or wrong |
| Code generation | ✅ Yes | Code passes tests or it doesn't |
| Wellness advice (safety) | ✅ Yes | Did it mention allergens? |
| Creative writing quality | ❌ No | Subjective |
| "Is this response friendly?" | ❌ No | Subjective |

This is why RLVR works so well for reasoning tasks — and why DeepSeek-R1 used it to achieve remarkable math and coding performance.

### Applied to Wellness

For our Personal Wellness Assistant, we can define verifiable rules like:
- **Safety rule**: Never recommend foods the user is allergic to
- **Format rule**: Always structure advice with numbered steps
- **Scope rule**: Only answer wellness-related questions
- **Intensity rule**: Recommend beginner exercises for beginner users

## Task 3: Defining a Verifiable Reward Function for Wellness

A reward function takes a model's response and returns a score. For RLVR, we want the score to be **automatic and objective**.

Let's define a user profile and a set of verifiable rules for our wellness assistant.

In [ ]:
# Define a sample user profile
user_profile = {
    "name": "Sarah",
    "fitness_level": "beginner",
    "allergies": ["peanuts", "shellfish"],
    "conditions": ["bad knee"],
    "goals": ["improve sleep", "reduce stress"]
}

print("User profile loaded:")
print(json.dumps(user_profile, indent=2))

In [ ]:
def reward_no_allergens(response: str, user_profile: dict) -> float:
    """
    Verifiable reward: response must not mention any of the user's allergens.
    Returns 1.0 if safe, 0.0 if an allergen is mentioned.
    """
    response_lower = response.lower()
    for allergen in user_profile["allergies"]:
        if allergen.lower() in response_lower:
            return 0.0
    return 1.0


def reward_beginner_appropriate(response: str, user_profile: dict) -> float:
    """
    Verifiable reward: for beginner users, response should not recommend
    high-intensity exercises like HIIT, heavy lifting, or sprinting.
    Returns 1.0 if appropriate, 0.0 if not.
    """
    if user_profile["fitness_level"] != "beginner":
        return 1.0  # Rule only applies to beginners
    
    high_intensity_terms = ["hiit", "heavy lifting", "sprint", "max effort", "intense"]
    response_lower = response.lower()
    
    for term in high_intensity_terms:
        if term in response_lower:
            return 0.0
    return 1.0


def reward_has_structure(response: str, user_profile: dict) -> float:
    """
    Verifiable reward: response should contain numbered steps or bullet points.
    Returns 1.0 if structured, 0.0 if not.
    """
    has_numbered = bool(re.search(r'\d+\.', response))
    has_bullets = bool(re.search(r'^[-*•]', response, re.MULTILINE))
    return 1.0 if (has_numbered or has_bullets) else 0.0


def compute_total_reward(response: str, user_profile: dict) -> dict:
    """
    Compute all rewards and return a summary.
    Total reward is the average of all individual rewards.
    """
    rewards = {
        "no_allergens": reward_no_allergens(response, user_profile),
        "beginner_appropriate": reward_beginner_appropriate(response, user_profile),
        "has_structure": reward_has_structure(response, user_profile),
    }
    rewards["total"] = sum(rewards.values()) / len(rewards)
    return rewards


print("Reward functions defined!")

## Task 4: Scoring Model Outputs

Let's test our reward functions on some example responses to make sure they work correctly before we use them for training.

In [ ]:
# Example responses to score
good_response = """
Here are some gentle exercises perfect for beginners:
1. Walking for 20-30 minutes
2. Swimming (easy on the knees)
3. Light yoga stretches

For snacks, try:
- Greek yogurt with berries
- Apple slices with sunflower seed butter
"""

bad_response_allergen = """
Here are some great snacks for energy:
1. Peanut butter on toast
2. Mixed nuts
3. Shrimp stir fry
"""

bad_response_intensity = """
To improve your fitness, try:
1. HIIT training 5 days a week
2. Heavy lifting sessions
3. Sprint intervals
"""

unstructured_response = "Just try walking more and eating better and getting enough sleep."

# Score each response
responses = {
    "Good response": good_response,
    "Bad (allergen)": bad_response_allergen,
    "Bad (too intense)": bad_response_intensity,
    "Unstructured": unstructured_response,
}

for name, response in responses.items():
    rewards = compute_total_reward(response, user_profile)
    print(f"\n{name}:")
    for k, v in rewards.items():
        print(f"  {k}: {v:.2f}")

## Task 5: The GRPO Training Loop

**GRPO (Group Relative Policy Optimization)** is the training algorithm used in DeepSeek-R1. It's a variant of PPO designed specifically for LLM fine-tuning with verifiable rewards.

### How It Works

```
For each training step:
  1. Sample a prompt (e.g., "What should Sarah eat for breakfast?")
  2. Generate G responses from the current model (rollouts)
  3. Score each response with the reward function
  4. Compute the advantage: how much better is this response vs the group average?
  5. Update the model to make high-reward responses more likely
  6. Repeat
```

### Key Terms

| Term | Meaning |
|------|---------|
| **Policy** | The language model we're training |
| **Rollout** | A generated response from the current policy |
| **Reward** | The score from our verifiable reward function |
| **Advantage** | How much better this rollout was vs average |
| **KL penalty** | Prevents the model from drifting too far from the original |

### Why GRPO over PPO?

Standard PPO requires a separate **value model** (critic) to estimate how good a state is. GRPO eliminates this by comparing responses within a group — if you generate 8 responses to the same prompt, the advantage of each is just how its reward compares to the group mean. This is much more memory efficient, which matters when your policy is a 7B+ parameter LLM.

---
## ❓ Question #1

What are the limitations of verifiable rewards for a wellness assistant? What kinds of wellness advice **cannot** be evaluated with a simple rule?

Consider:
- Empathy and tone
- Nuanced medical advice
- Cultural sensitivity
- Motivational quality

#### Answer:
*Your answer here*

## ❓ Question #2

What is **reward hacking**? Give a concrete example of how a wellness assistant might hack the reward functions we defined in Task 3.

Consider:
- A model that gets a perfect score without giving useful advice
- How you might detect and prevent this

#### Answer:
*Your answer here*

---
## 🏗️ Activity #1: Build and Test a Wellness Reward Function

Build your own verifiable reward function for the wellness assistant. Your function should check something objective and automatically scoreable.

### Requirements:
- Define at least one new reward function not already in Task 3
- Test it on at least 3 example responses (one that passes, one that fails, one edge case)
- Integrate it into `compute_total_reward`
- Explain in a markdown cell why your reward is verifiable

In [ ]:
### YOUR CODE HERE ###

def reward_your_function(response: str, user_profile: dict) -> float:
    """
    TODO: Define your own verifiable reward function.
    """
    pass


# Test your reward function on at least 3 responses
test_responses = [
    "Response that should pass...",
    "Response that should fail...",
    "Edge case response...",
]

for response in test_responses:
    score = reward_your_function(response, user_profile)
    print(f"Score: {score:.2f} | Response: {response[:60]}...")

---
# 🤝 Breakout Room #2
## Training and Evaluation

## Task 6: Setting Up a Small Language Model

We'll use **Qwen 2.5 1.5B** — a small but capable model that can run on a free Colab T4 GPU. This is a proof-of-concept training loop; in production you'd use a larger model and more data.

We're using a small model here intentionally:
- Trains in minutes, not days
- Fits on a free GPU
- Still demonstrates the RLVR concept clearly

In [ ]:
# Load model and tokenizer
# Note: This will download ~3GB on first run
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Task 7: Generating Rollouts

A **rollout** is a response generated by the current model. In GRPO, we generate multiple rollouts per prompt so we can compare them and compute relative advantages.

In [ ]:
def generate_rollouts(prompt: str, model, tokenizer, num_rollouts: int = 4) -> list[str]:
    """
    Generate multiple responses to the same prompt.
    These are our 'rollouts' for GRPO.
    """
    messages = [
        {"role": "system", "content": f"You are a Personal Wellness Assistant helping a user with profile: {json.dumps(user_profile)}"},
        {"role": "user", "content": prompt}
    ]
    
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    responses = []
    for _ in range(num_rollouts):
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=True,
                temperature=0.8,
                pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(
            output[0][inputs['input_ids'].shape[1]:], 
            skip_special_tokens=True
        )
        responses.append(response)
    
    return responses


# Test rollout generation
test_prompt = "What should I eat for breakfast to improve my energy levels?"
print(f"Generating rollouts for: '{test_prompt}'\n")

rollouts = generate_rollouts(test_prompt, model, tokenizer, num_rollouts=2)
for i, rollout in enumerate(rollouts):
    print(f"--- Rollout {i+1} ---")
    print(rollout)
    rewards = compute_total_reward(rollout, user_profile)
    print(f"Total reward: {rewards['total']:.2f}\n")

## Task 8: Computing Rewards and Advantages

The **advantage** tells us how much better a response is compared to the average response for that prompt. This is the key innovation of GRPO — instead of needing a value model, we just compare within the group.

In [ ]:
def compute_advantages(rollouts: list[str], user_profile: dict) -> tuple[list[float], list[float]]:
    """
    Compute rewards and advantages for a group of rollouts.
    
    Advantage = reward - mean(rewards)
    This tells us: how much better is this response vs the average?
    """
    # Score each rollout
    rewards = [compute_total_reward(r, user_profile)["total"] for r in rollouts]
    
    # Compute group mean and std
    mean_reward = sum(rewards) / len(rewards)
    std_reward = (sum((r - mean_reward) ** 2 for r in rewards) / len(rewards)) ** 0.5
    
    # Normalize advantages
    if std_reward > 0:
        advantages = [(r - mean_reward) / (std_reward + 1e-8) for r in rewards]
    else:
        advantages = [0.0] * len(rewards)
    
    return rewards, advantages


# Demonstrate with our rollouts
rewards, advantages = compute_advantages(rollouts, user_profile)

print("Rewards and Advantages:")
for i, (r, a) in enumerate(zip(rewards, advantages)):
    print(f"  Rollout {i+1}: reward={r:.2f}, advantage={a:.2f}")
print(f"\nMean reward: {sum(rewards)/len(rewards):.2f}")
print("\nPositive advantage → model should do MORE of this")
print("Negative advantage → model should do LESS of this")

## Task 9: Running a GRPO Update Step

We'll use HuggingFace TRL's `GRPOTrainer` to handle the actual training loop. It wraps all the complexity of GRPO (policy gradient updates, KL penalties, advantage computation) into a clean API.

We define a **reward function** that TRL calls automatically during training.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Build a small training dataset of wellness prompts
wellness_prompts = [
    {"prompt": "What should I eat for breakfast to boost my energy?"},
    {"prompt": "What exercises do you recommend for me?"},
    {"prompt": "How can I improve my sleep quality?"},
    {"prompt": "What's a good afternoon snack to keep me going?"},
    {"prompt": "How do I reduce stress after work?"},
    {"prompt": "What stretches can I do at home?"},
    {"prompt": "How much water should I drink daily?"},
    {"prompt": "What foods should I eat to reduce inflammation?"},
]

# Format prompts with system context
def format_prompt(example):
    messages = [
        {"role": "system", "content": f"You are a Personal Wellness Assistant. User profile: {json.dumps(user_profile)}"},
        {"role": "user", "content": example["prompt"]}
    ]
    example["prompt"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return example

dataset = Dataset.from_list(wellness_prompts)
dataset = dataset.map(format_prompt)

print(f"Training dataset: {len(dataset)} prompts")

In [ ]:
# Define the reward function for TRL
# TRL calls this with a list of completions and expects a list of floats
def wellness_reward_function(completions: list[str], **kwargs) -> list[float]:
    """
    Reward function compatible with TRL's GRPOTrainer.
    Returns a reward score for each completion.
    """
    rewards = []
    for completion in completions:
        result = compute_total_reward(completion, user_profile)
        rewards.append(result["total"])
    return rewards


# Configure GRPO training
# Note: Using minimal settings for a proof-of-concept run
grpo_config = GRPOConfig(
    output_dir="./rlvr_wellness_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_generations=4,       # Number of rollouts per prompt (G in GRPO)
    max_new_tokens=200,
    max_prompt_length=512,
    temperature=0.8,
    logging_steps=1,
    report_to="none",        # Set to "wandb" for experiment tracking
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=wellness_reward_function,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("GRPOTrainer initialized!")
print(f"Training on {len(dataset)} prompts with {grpo_config.num_generations} rollouts each")

In [ ]:
# Run training
# Note: Even 1 epoch on 8 prompts is enough to see reward improvement
print("Starting GRPO training...")
trainer.train()
print("Training complete!")

## Task 10: Evaluating Before and After Training

Let's compare the base model's responses to the RLVR-trained model's responses on the same prompts. We should see higher reward scores after training.

In [ ]:
def evaluate_model(model, tokenizer, prompts: list[str], user_profile: dict) -> dict:
    """
    Evaluate a model on a set of prompts and return average reward scores.
    """
    all_rewards = {"no_allergens": [], "beginner_appropriate": [], "has_structure": [], "total": []}
    
    for prompt in prompts:
        rollouts = generate_rollouts(prompt, model, tokenizer, num_rollouts=2)
        for rollout in rollouts:
            rewards = compute_total_reward(rollout, user_profile)
            for key in all_rewards:
                all_rewards[key].append(rewards[key])
    
    return {k: sum(v) / len(v) for k, v in all_rewards.items()}


eval_prompts = [
    "What should I eat for breakfast?",
    "What exercises do you recommend?",
    "How can I improve my sleep?"
]

print("Evaluating trained model...")
trained_results = evaluate_model(model, tokenizer, eval_prompts, user_profile)

print("\n=== Evaluation Results ===")
print("\nTrained Model:")
for k, v in trained_results.items():
    print(f"  {k}: {v:.3f}")

print("\nNote: Compare these scores to the base model outputs from Task 7.")
print("After training, we expect higher scores on our verifiable reward criteria.")

---
## ❓ Question #3

Look at the reward scores before and after training. Did the model improve on all three reward criteria equally? Why might some rewards be easier to optimize than others?

Consider:
- The specificity of each reward function
- Whether the model had already learned these patterns from pretraining
- The size of our training dataset

#### Answer:
*Your answer here*

## ❓ Question #4

How would you scale this RLVR setup for a **production wellness assistant**? What would need to change?

Consider:
- Model size
- Dataset size and diversity
- Number and complexity of reward functions
- Handling edge cases and reward hacking
- Continuous learning as user profiles change

#### Answer:
*Your answer here*

---
## 🏗️ Activity #2: Compare Base vs RLVR-Trained Wellness Responses

Build a side-by-side comparison of base model vs trained model responses.

### Requirements:
- Test both models on at least 3 new prompts not seen during training
- Score both sets of responses with `compute_total_reward`
- Display results in a clear comparison table
- Write a short analysis: where did RLVR help most? Where did it fail?

In [ ]:
### YOUR CODE HERE ###

# Hint: Load the base model separately, or save responses before training
# Then compare against trained model responses

new_prompts = [
    # Add at least 3 prompts here
]

# Generate and score responses from trained model
# Compare to base model responses
# Display comparison table

---
## Summary

In this session, we explored **RLVR** as an alternative to RLHF:

| Concept | Key Takeaway |
|---------|-------------|
| **Verifiable rewards** | Replace human raters with automatic rule-based functions |
| **GRPO** | Compare rollouts within a group to compute advantages — no value model needed |
| **Reward hacking** | Models can exploit poorly designed reward functions — design carefully |
| **Evaluation** | Always compare before and after training on held-out prompts |
| **Limitations** | RLVR only works when correctness is objectively measurable |

### Key Takeaways:

1. **Verifiable rewards are powerful but limited** — they work best for objective, rule-based tasks
2. **GRPO is efficient** — by comparing within groups, it avoids the need for a separate value model
3. **Reward design is the hard part** — getting the reward function right matters more than the training algorithm
4. **Small models can demonstrate the concept** — but production use requires larger models and more data
5. **RLVR + RLHF can complement each other** — use verifiable rewards where possible, human feedback where not

### Further Reading:

- [DeepSeek-R1 Paper](https://arxiv.org/abs/2501.12948) — the flagship RLVR model
- [TRL Documentation](https://huggingface.co/docs/trl/en/grpo_trainer) — GRPOTrainer docs
- [GRPO Paper](https://arxiv.org/abs/2402.03300) — original GRPO algorithm
- [Reward Hacking Survey](https://arxiv.org/abs/1906.01820) — understanding reward hacking